# Markov Chains and Mixing Times

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 09.02 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/03_markov_chains_and_mixing.ipynb)

---

## 🎯 Learning Objective

Study transition matrices, stationary distributions, and mixing times, and understand ergodicity.

---

## 📐 Theory

`09.02` studied Brownian motion, a *continuous*-state, *continuous*-time Markov process. This
notebook studies the discrete-state, discrete-time case — simpler to compute with exactly, and
the direct engine behind MCMC (`03.09`), PageRank, and autoregressive text generation.

### Markov chains and transition matrices

A **Markov chain** is a sequence of random variables $X_0, X_1, X_2, \ldots$ taking values in a
finite state space $\{1,\ldots,n\}$, satisfying the **Markov property**: the future depends on
the past only through the present,

$$P(X_{t+1} = j \mid X_t = i, X_{t-1}, \ldots, X_0) = P(X_{t+1} = j \mid X_t = i) =: P_{ij}.$$

Collecting these one-step probabilities into an $n\times n$ **transition matrix** $P$ (row $i$ =
the distribution of where you go next, given you're at state $i$; every row sums to $1$ — a
**row-stochastic** matrix) turns the entire dynamics into linear algebra. If $\mu_t$ is the
(row-vector) distribution of $X_t$ over states, then $\mu_{t+1} = \mu_t P$ — one matrix
multiplication advances the whole distribution by one step, and $\mu_t = \mu_0 P^t$ jumps
straight to $t$ steps ahead. This is the discrete-time, discrete-state analogue of the SDE
machinery from `09.02`; a Wiener process is, in a sense, what you get taking the state space
continuous and the time step to zero.

### Stationary distributions

A distribution $\pi$ (row vector, entries summing to $1$) is **stationary** for $P$ if
$\pi P = \pi$ — once you reach $\pi$, every subsequent step leaves the distribution unchanged
(individual states still hop around; the *population-level* distribution over states is fixed).
Since $\pi P = \pi$ means $\pi P^\top{}^\top = \pi$, i.e. $\pi^\top$ is a genuine eigenvector of
$P^\top$ with eigenvalue $1$, computing $\pi$ reduces to eigendecomposition (`01.06`) — one of two
methods this notebook implements and cross-checks against each other. The **Perron-Frobenius
theorem** guarantees that for any row-stochastic $P$, an eigenvalue-$1$ eigenvector with
non-negative entries always exists, and if the chain is **irreducible** (every state reachable
from every other) it is unique up to scaling — so normalizing it to sum to $1$ pins down $\pi$.

### Ergodicity and convergence to stationarity

A chain is **ergodic** if it is irreducible and **aperiodic** (it doesn't cycle through states on
a fixed rhythm — having any state with a self-loop, i.e. $P_{ii}>0$, already rules out
periodicity). The **ergodic theorem for Markov chains** then guarantees two independently useful
facts, both verified numerically below: (1) *regardless of the starting distribution* $\mu_0$,
the chain's distribution $\mu_0 P^t \to \pi$ as $t\to\infty$ — so $\pi$ can be estimated by
simply running the chain for a long time and recording how often each state is visited, no
eigendecomposition needed; and (2) time averages converge to the stationary expectation almost
surely (`09.01`'s language) — the Markov-chain analogue of the Law of Large Numbers, and exactly
what justifies treating MCMC samples (`03.09`) as representative draws from the target
distribution once the chain has run long enough.

### Mixing time: how long is "long enough"?

**Total variation (TV) distance** between two distributions on the same finite state space is
$\|\mu-\nu\|_{TV} = \frac12\sum_i |\mu_i - \nu_i|$ — half the $L^1$ distance between them, always
between $0$ (identical) and $1$ (disjoint support). The **mixing time** $t_{\text{mix}}(\epsilon)$
is the first time $t$ such that $\|\mu_0 P^t - \pi\|_{TV} < \epsilon$ for every possible starting
distribution $\mu_0$ — the point at which the chain has (nearly) forgotten where it started. For
an ergodic chain, $\|\mu_0 P^t - \pi\|_{TV}$ decays *exponentially* in $t$, at a rate governed by
the transition matrix's **second-largest eigenvalue modulus (SLEM)**, $|\lambda_2|$ (the
largest-magnitude eigenvalue other than the guaranteed $\lambda_1=1$): roughly,
$\|\mu_0 P^t - \pi\|_{TV} \propto |\lambda_2|^t$. A SLEM close to $1$ means slow mixing (the chain
has some near-stationary direction it escapes only slowly, often because of a "bottleneck"
separating two weakly-connected clusters of states); a small SLEM means fast mixing. This
spectral-gap-controls-convergence-speed pattern is a discrete, finite-state preview of the same
idea behind how MCMC samplers (`03.09`) are diagnosed for "have I run this long enough" in
practice.

### PageRank: a stationary distribution, applied to the entire web

**PageRank** models a "random surfer" clicking links: from any web page, follow an outgoing link
uniformly at random (with a small probability $1-d$ of "teleporting" to a uniformly random page
instead, guaranteeing irreducibility even if the link graph itself isn't strongly connected).
This defines a genuine Markov chain — the **Google matrix**
$G = d\,P + (1-d)\,\tfrac{1}{n}\mathbf{1}\mathbf{1}^\top$, where $P$ is the link-following
transition matrix and $d\approx 0.85$ — and a page's **PageRank score** is defined to be exactly
its stationary probability $\pi_i$: the long-run fraction of clicks that land on page $i$. This
notebook implements PageRank as nothing more than a literal instance of the stationary
distribution computed above, on a graph built with the same `networkx` tools as `07.02`.

---

In [ ]:
# Setup
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting total variation distance to stationarity over time, for a well-connected chain versus
one with a weak bottleneck between two clusters of states, makes "mixing time" and "spectral
gap" visible as the same phenomenon: the bottlenecked chain's TV distance decays exponentially,
but at a visibly slower rate.

In [ ]:
# Two Markov chains, same recipe for computing stationarity, very different mixing speed:
# a well-connected 5-state chain vs. a 6-state chain with two clusters joined by a weak bridge.
rng = np.random.default_rng(0)

def stationary_via_eigenvector(P):
    eigvals, eigvecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(eigvals - 1))    # the eigenvalue guaranteed to be exactly 1 (Perron-Frobenius)
    pi = np.real(eigvecs[:, idx])
    return pi / pi.sum()

def tv_distance(p, q):
    return 0.5 * np.sum(np.abs(p - q))

# Well-connected chain: random transitions, no obvious bottleneck
n_fast = 5
raw = rng.uniform(0.1, 1.0, size=(n_fast, n_fast))
P_fast = raw / raw.sum(axis=1, keepdims=True)
pi_fast = stationary_via_eigenvector(P_fast)

# Bottlenecked chain: two 3-state clusters, strong within-cluster transitions, weak across
n_slow = 6
P_slow = np.zeros((n_slow, n_slow))
within, cross = 0.30, 0.02
for i in range(n_slow):
    cluster_i = 0 if i < 3 else 1
    for j in range(n_slow):
        cluster_j = 0 if j < 3 else 1
        if i == j:
            P_slow[i, j] = 0.5             # "lazy" self-loop, also guarantees aperiodicity
        elif cluster_j == cluster_i:
            P_slow[i, j] = within
        else:
            P_slow[i, j] = cross            # rare cross-cluster jump = the bottleneck
    P_slow[i] /= P_slow[i].sum()
pi_slow = stationary_via_eigenvector(P_slow)

def tv_curve(P, pi, n_iters, start_state=0):
    dist = np.zeros(len(pi)); dist[start_state] = 1.0
    curve = []
    for _ in range(n_iters):
        curve.append(tv_distance(dist, pi))
        dist = dist @ P
    return np.array(curve)

curve_fast = tv_curve(P_fast, pi_fast, 30)
curve_slow = tv_curve(P_slow, pi_slow, 60)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(curve_fast, "o-", color="#4C72B0", markersize=4, label="well-connected 5-state chain")
ax.semilogy(curve_slow, "s-", color="#C44E52", markersize=4, label="bottlenecked 6-state chain (2 clusters)")
ax.axhline(0.25, color="gray", ls="--", lw=1, label="eps=0.25 threshold")
ax.set_xlabel("t (steps)"); ax.set_ylabel("TV distance to stationary pi  (log scale)")
ax.set_title("Mixing time: TV distance decays EXPONENTIALLY,\nbut a bottleneck makes the rate much slower")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

eps = 0.25
t_mix_fast = next(t for t, v in enumerate(curve_fast) if v < eps)
t_mix_slow = next(t for t, v in enumerate(curve_slow) if v < eps)
print(f"t_mix(0.25) for the well-connected chain: {t_mix_fast} steps")
print(f"t_mix(0.25) for the bottlenecked chain:   {t_mix_slow} steps")
print(f"The bottlenecked chain takes {t_mix_slow/t_mix_fast:.1f}x longer to mix -- a direct,")
print("visible consequence of the weak cross-cluster transition probability, quantified")
print("rigorously below via the transition matrix's second-largest eigenvalue.")

## 🐍 Implementation from Scratch

We compute a Markov chain's stationary distribution two independent ways (eigenvector of $P^\top$
vs. long-run empirical simulation) and confirm they agree, then verify that the total variation
decay rate matches the transition matrix's second-largest eigenvalue exactly — not just
approximately — before treating that as a general "mixing rate" fact.

In [ ]:
# --- Stationary distribution, computed TWO independent ways ---
def stationary_via_eigenvector(P):
    # pi P = pi means pi^T is an eigenvector of P^T with eigenvalue 1 (Perron-Frobenius
    # guarantees this eigenvalue exists and, for an irreducible chain, is unique up to scale)
    eigvals, eigvecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(eigvals - 1))
    pi = np.real(eigvecs[:, idx])
    return pi / pi.sum()

def stationary_via_simulation(P, n_steps, rng, burn_in=1000):
    # The ergodic theorem's OTHER guarantee: long-run visit frequencies converge to pi,
    # regardless of start state -- no eigendecomposition needed, just running the chain.
    n = len(P)
    state = 0
    counts = np.zeros(n)
    for t in range(n_steps + burn_in):
        state = rng.choice(n, p=P[state])
        if t >= burn_in:            # discard the transient before the chain has "forgotten" state 0
            counts[state] += 1
    return counts / n_steps

rng = np.random.default_rng(0)
n = 5
raw = rng.uniform(0.1, 1.0, size=(n, n))
P = raw / raw.sum(axis=1, keepdims=True)

pi_eig = stationary_via_eigenvector(P)
pi_sim = stationary_via_simulation(P, n_steps=300_000, rng=np.random.default_rng(1))
print("Stationary distribution via eigenvector: ", np.round(pi_eig, 4))
print("Stationary distribution via simulation:  ", np.round(pi_sim, 4))
print(f"Max absolute difference: {np.max(np.abs(pi_eig - pi_sim)):.5f}  (two independent methods, close agreement)")
print(f"Sanity check pi @ P == pi (eigenvector method): {np.allclose(pi_eig @ P, pi_eig, atol=1e-8)}")

# --- Mixing time and the spectral gap: does the TV decay rate ACTUALLY equal the SLEM, or is
# that just a plausible-sounding rule of thumb? Fit the decay rate directly and compare. ---
n_slow = 6
P_slow = np.zeros((n_slow, n_slow))
within, cross = 0.30, 0.02
for i in range(n_slow):
    cluster_i = 0 if i < 3 else 1
    for j in range(n_slow):
        cluster_j = 0 if j < 3 else 1
        if i == j:
            P_slow[i, j] = 0.5
        elif cluster_j == cluster_i:
            P_slow[i, j] = within
        else:
            P_slow[i, j] = cross
    P_slow[i] /= P_slow[i].sum()
pi_slow = stationary_via_eigenvector(P_slow)

def tv_distance(p, q):
    return 0.5 * np.sum(np.abs(p - q))

dist = np.zeros(n_slow); dist[0] = 1.0
tv_curve = []
for _ in range(60):
    tv_curve.append(tv_distance(dist, pi_slow))
    dist = dist @ P_slow
tv_curve = np.array(tv_curve)

eigvals_slow = np.linalg.eigvals(P_slow)
eigvals_sorted = sorted(eigvals_slow, key=lambda x: -abs(x))
slem = abs(eigvals_sorted[1])   # second-largest eigenvalue MODULUS (the largest one besides 1)

# Fit log(TV(t)) ~ t*log(rate) + const over an ASYMPTOTIC window (skip the early transient,
# where sub-dominant eigenvalues besides the SLEM still contribute) -- an independent estimate
# of the true decay rate, to check against the SLEM rather than assume the match.
fit_window = np.arange(15, 45)
slope, intercept = np.polyfit(fit_window, np.log(tv_curve[fit_window]), 1)
fitted_rate = np.exp(slope)

print(f"\nSecond-largest eigenvalue modulus (SLEM) of P_slow: {slem:.6f}")
print(f"TV-distance decay rate, fit independently from the simulated curve: {fitted_rate:.6f}")
print(f"Match: {np.isclose(fitted_rate, slem, rtol=1e-3)}  "
      f"(confirms mixing speed IS governed by the spectral gap 1-|lambda_2|, not a coincidence)")

## 🤖 Why This Matters for AI

MCMC convergence diagnostics, PageRank, and autoregressive text generation itself is a (very
high-order) Markov chain. PageRank is the cleanest possible example: a page's importance score
*is*, by definition, its stationary probability under a "random surfer" Markov chain over the
web graph (`07.02`'s adjacency-matrix language, applied to hyperlinks). We implement it below as
nothing more than an instance of `stationary_via_eigenvector`, and cross-check it against
`networkx.pagerank`'s independent implementation and against power iteration (repeatedly
multiplying by the transition matrix) — the same three-way check used above for the toy chain,
now applied to a genuine graph problem.

In [ ]:
# PageRank = the stationary distribution of a "random surfer" Markov chain over a link graph.
# Not an analogy -- this cell computes it as a LITERAL instance of stationary_via_eigenvector.
rng = np.random.default_rng(3)
G = nx.gnp_random_graph(8, 0.3, directed=True, seed=3)
G.remove_edges_from(nx.selfloop_edges(G))   # a page linking to itself isn't a meaningful "vote"

n_pages = G.number_of_nodes()
A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
out_degree = A.sum(axis=1)

damping = 0.85   # standard PageRank damping factor
P_link = np.zeros((n_pages, n_pages))
for i in range(n_pages):
    if out_degree[i] == 0:
        P_link[i, :] = 1.0 / n_pages     # dangling page (no outlinks): teleport uniformly
    else:
        P_link[i, :] = A[i, :] / out_degree[i]   # follow an outgoing link uniformly at random

# The Google matrix: with probability `damping` follow a real link, else teleport uniformly --
# the teleport term is what GUARANTEES irreducibility (every page reachable from every other),
# which is exactly the condition Perron-Frobenius needs for a unique stationary distribution.
google_matrix = damping * P_link + (1 - damping) * np.ones((n_pages, n_pages)) / n_pages
print(f"Google matrix row sums (must all be 1): {np.round(google_matrix.sum(axis=1), 6)}")

pagerank_eig = stationary_via_eigenvector(google_matrix)

# Independent check #1: networkx's OWN pagerank implementation
pagerank_nx = nx.pagerank(G, alpha=damping)
pagerank_nx_arr = np.array([pagerank_nx[i] for i in sorted(G.nodes())])

# Independent check #2: power iteration (repeated multiplication by the Google matrix) --
# the actual algorithm real search engines used historically, requiring no eigendecomposition
v = np.ones(n_pages) / n_pages
for _ in range(300):
    v = v @ google_matrix
pagerank_power_iter = v

print(f"\n{'page':>5} | {'via eigenvector':>16} | {'via networkx':>14} | {'via power iteration':>20}")
for i in range(n_pages):
    print(f"{i:5d} | {pagerank_eig[i]:16.4f} | {pagerank_nx_arr[i]:14.4f} | {pagerank_power_iter[i]:20.4f}")

print(f"\nMax diff (eigenvector vs networkx):      {np.max(np.abs(pagerank_eig - pagerank_nx_arr)):.2e}")
print(f"Max diff (eigenvector vs power iteration): {np.max(np.abs(pagerank_eig - pagerank_power_iter)):.2e}")

top_page = np.argmax(pagerank_eig)
in_degrees = A.sum(axis=0)
print(f"\nHighest-ranked page: {top_page}, PageRank={pagerank_eig[top_page]:.4f}, "
      f"in-degree={int(in_degrees[top_page])} (max in-degree across all pages: {int(in_degrees.max())})")
print("PageRank score tracks being LINKED TO (in-degree), not how many links a page sends out --")
print("the random surfer's stationary probability accumulates wherever incoming paths converge.")

## 🏋️ Exercises

### Warm-up
1. For the 2-state chain $P = \begin{pmatrix}0.9 & 0.1\\ 0.4 & 0.6\end{pmatrix}$, solve
   $\pi P = \pi$ by hand (using $\pi_1+\pi_2=1$) to find the stationary distribution. Verify
   with `stationary_via_eigenvector`.

### Practice
2. In the mixing-time experiment, increase `cross` from `0.02` to `0.10` (a less severe
   bottleneck) and recompute the SLEM and the fitted TV-decay rate. Confirm they still match
   each other, and confirm $t_{\text{mix}}(0.25)$ is now smaller than the original bottlenecked
   chain's. Explain the relationship between the bottleneck's strength and the spectral gap
   $1-|\lambda_2|$.

### Challenge
3. Autoregressive language models generate token $x_{t+1}$ from a distribution conditioned on
   the *entire* history $x_1,\ldots,x_t$, seemingly violating the Markov property (which only
   allows dependence on $x_t$). Argue why treating the full history "so far" as a single Markov
   *state* $s_t = (x_1,\ldots,x_t)$ restores the Markov property trivially, and explain (in 2-3
   sentences) why this trivial fix doesn't help in practice — connect your answer to why
   real language models use a fixed-size hidden state or attention over a context window instead
   of a literally growing state space. Then, in the PageRank graph, remove node `4` (a dangling
   page) entirely and recompute PageRank for the remaining 7 pages — does removing a
   zero-out-degree node change the *relative order* of the remaining pages' scores?

---

## 📚 Further Reading
- Levin & Peres, *Markov Chains and Mixing Times*, Ch. 1, 4, 12 (the standard modern reference)
- Page, Brin, Motwani & Winograd, ["The PageRank Citation Ranking"](http://ilpubs.stanford.edu:8090/422/) (1999)
- Grinstead & Snell, *Introduction to Probability*, Ch. 11 (Markov Chains) — free online

---

*Next notebook: [Concentration Inequalities](04_concentration_inequalities.ipynb)*